# Limpeza, Preparação e Armazenamento de Dados — NFL Play by Play

## Decisões de tratamento

- Campos de contexto de uma jogada podem estar ausentes por não se aplicarem ao evento ou por não terem sido registrados. Por isso, não serão preenchidos por média/moda nem serão removidas linhas apenas por conterem valores ausentes.
- Strings vazias ou compostas somente por espaços serão padronizadas como ausentes; espaços no início/fim dos textos serão removidos.
- `Date` será interpretada como data; colunas numéricas e indicadores serão convertidos somente a partir de listas explícitas de atributos conhecidos, sem inferir que identificadores/códigos textuais sejam números.
- Indicadores só serão convertidos para booleano anulável quando seus valores observados forem exclusivamente 0 e 1. Valores fora desse domínio serão reportados e preservados.
- Somente linhas integralmente duplicadas em todas as colunas serão removidas. Possíveis anomalias de domínio serão quantificadas, mas não corrigidas nem excluídas sem regra de negócio que justifique a alteração.

In [1]:
# Importa as bibliotecas usadas
from pathlib import Path
import numpy as np
import pandas as pd
import kagglehub
from pandas.testing import assert_frame_equal

c:\Users\41228408807\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carregar a fonte de dado

O dataset é obtido pelo mesmo identificador KaggleHub utilizado no notebook `Pre_processamento_dados_Gabarito.ipynb`. O caminho é localizado pelo nome do CSV para não depender de um caminho local específico.

In [2]:
# Baixa/localiza a versão do dataset e encontra o arquivo esperado
dataset_root = Path(kagglehub.dataset_download("maxhorowitz/nflplaybyplay2009to2016"))
csv_name = "NFL Play by Play 2009-2017 (v4).csv"
csv_matches = list(dataset_root.rglob(csv_name))
if not csv_matches:
    raise FileNotFoundError(f"Arquivo {csv_name!r} não encontrado em {dataset_root}")
csv_path = csv_matches[0]

# Lê sem conversão parcial de tipos para reduzir avisos de tipos mistos.
df_original = pd.read_csv(csv_path, low_memory=False)
df = df_original.copy()
print(f"CSV: {csv_path}")
print(f"Dimensões originais: {df.shape[0]:,} linhas × {df.shape[1]:,} colunas")

CSV: C:\Users\41228408807\.cache\kagglehub\datasets\maxhorowitz\nflplaybyplay2009to2016\versions\6\NFL Play by Play 2009-2017 (v4).csv
Dimensões originais: 407,688 linhas × 102 colunas


## 2. Diagnóstico

As saídas abaixo mostram a estrutura, tipos inferidos na leitura, estatísticas e completude por atributo. O levantamento de strings vazias complementa `isna()`, pois uma célula vazia em um CSV nem sempre é representada como ausente pelo leitor.

In [3]:
# Inspeciona o df, estrutura e tipos
display(df.head())
df.info(verbose=True)
display(df.describe(include="all").T)
display(df.dtypes.value_counts().rename("quantidade de colunas").to_frame())

,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


<class 'pandas.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Data columns (total 102 columns):
 #    Column                       Dtype  
---   ------                       -----  
 0    Date                         str    
 1    GameID                       int64  
 2    Drive                        int64  
 3    qtr                          int64  
 4    down                         float64
 5    time                         str    
 6    TimeUnder                    int64  
 7    TimeSecs                     float64
 8    PlayTimeDiff                 float64
 9    SideofField                  str    
 10   yrdln                        float64
 11   yrdline100                   float64
 12   ydstogo                      int64  
 13   ydsnet                       int64  
 14   GoalToGo                     float64
 15   FirstDown                    float64
 16   posteam                      str    
 17   DefensiveTeam                str    
 18   desc                         str 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Date,407688,426,2016-01-03,2872,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GameID,407688.0,NaN,NaN,NaN,2013157548.708797,2572838.560767,2009091000.0,2011100907.0,2013111000.0,2015121301.0,2017123115.0
Drive,407688.0,NaN,NaN,NaN,12.316158,7.149527,1.0,6.0,12.0,18.0,35.0
qtr,407688.0,NaN,NaN,NaN,2.577412,1.12975,1.0,2.0,3.0,4.0,5.0
down,346534.0,NaN,NaN,NaN,2.002476,1.006353,1.0,1.0,2.0,3.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
Win_Prob,382679.0,NaN,NaN,NaN,0.50132,0.287445,0.0,0.276472,0.50447,0.725477,1.0
WPA,402147.0,NaN,NaN,NaN,0.002099,0.045363,-0.997214,-0.014728,0.0,0.014684,0.994848
airWPA,159187.0,NaN,NaN,NaN,0.015135,0.05649,-0.999881,-0.011518,0.003441,0.035792,0.994848
yacWPA,158926.0,NaN,NaN,NaN,-0.01048,0.068139,-0.986673,-0.018683,0.0,0.011431,1.0


,quantidade de colunas
str,38
float64,33
int64,31


In [4]:
# Resume valores ausentes, incluindo proporção por coluna.
missing_initial = pd.DataFrame({
    "ausentes": df.isna().sum(),
    "percentual": df.isna().mean().mul(100),
}).sort_values("ausentes", ascending=False)
display(missing_initial)

,ausentes,percentual
DefTwoPoint,407664,99.994113
BlockingPlayer,407571,99.971302
TwoPointConv,407083,99.851602
ChalReplayResult,404286,99.165538
RecFumbPlayer,403315,98.927366
...,...,...
HomeTeam,0,0.000000
Timeout_Indicator,0,0.000000
TwoPoint_Prob,0,0.000000
ExPoint_Prob,0,0.000000


In [5]:
# Conta textos vazios ou compostos somente por espaços nas colunas textuais.
text_columns_initial = df.select_dtypes(include=["object", "string"]).columns
blank_initial = df[text_columns_initial].apply(
    lambda column: column.astype("string").str.strip().eq("").sum()
)

display(blank_initial[blank_initial.gt(0)].rename("strings vazias/brancas").to_frame())

display(df.nunique(dropna=True).sort_values(ascending=False).rename("valores distintos").to_frame())
exact_duplicates_initial = int(df.duplicated().sum())

print(f"Linhas integralmente duplicadas: {exact_duplicates_initial:,}")

,strings vazias/brancas


,valores distintos
desc,375962
WPA,366549
Away_WP_post,363360
Home_WP_post,363190
Away_WP_pre,362562
...,...
ChalReplayResult,2
Timeout_Indicator,2
InterceptionThrown,2
TwoPoint_Prob,2


Linhas integralmente duplicadas: 0


## Nomes das colunas

A célula seguinte imprime cada nome de coluna, numerado e sem truncamento, para facilitar a elaboração de um dicionário de dados.

In [6]:
# Lista todos os nomes das colunas para consulta no dicionário de dados.
for position, column_name in enumerate(df.columns, start=1):
    print(f"{position:03d}. {column_name}")

001. Date
002. GameID
003. Drive
004. qtr
005. down
006. time
007. TimeUnder
008. TimeSecs
009. PlayTimeDiff
010. SideofField
011. yrdln
012. yrdline100
013. ydstogo
014. ydsnet
015. GoalToGo
016. FirstDown
017. posteam
018. DefensiveTeam
019. desc
020. PlayAttempted
021. Yards.Gained
022. sp
023. Touchdown
024. ExPointResult
025. TwoPointConv
026. DefTwoPoint
027. Safety
028. Onsidekick
029. PuntResult
030. PlayType
031. Passer
032. Passer_ID
033. PassAttempt
034. PassOutcome
035. PassLength
036. AirYards
037. YardsAfterCatch
038. QBHit
039. PassLocation
040. InterceptionThrown
041. Interceptor
042. Rusher
043. Rusher_ID
044. RushAttempt
045. RunLocation
046. RunGap
047. Receiver
048. Receiver_ID
049. Reception
050. ReturnResult
051. Returner
052. BlockingPlayer
053. Tackler1
054. Tackler2
055. FieldGoalResult
056. FieldGoalDistance
057. Fumble
058. RecFumbTeam
059. RecFumbPlayer
060. Sack
061. Challenge.Replay
062. ChalReplayResult
063. Accepted.Penalty
064. PenalizedTeam
065. Penalt

## 3. Tipificação e padronização textual

Cada coluna abaixo recebe o tipo mais estreito que ainda representa o valor observado e os ausentes, para ocupar menos espaço no Parquet.

- Inteiro sem ausentes: `int8`, `int16` ou `int32`.
- Inteiro com ausentes: o equivalente anulável (`Int8`, `Int16`, `Int32`).
- Indicador 0/1 sem ausentes: `bool` (bit no Parquet). Com ausentes: `boolean`.
- Decimal de modelo (probabilidade, EPA, WPA): `float32`, ou `Float32` quando há ausente. Esse tamanho guarda as casas dessas métricas.
- Texto de domínio pequeno (sigla, tipo de jogada, nome que se repete): `category`, gravado como dicionário.
- `desc`: `string`, porque o texto da súmula é livre e quase único por linha.
- `Date`: data do jogo. `time`: o relógio `MM:SS` vira segundos em `Int16`.

A primeira célula só padroniza espaços e textos vazios. Cada coluna seguinte tem a própria célula. Se aparecer um valor fora de 0/1 num indicador, a coluna fica como está e o caso entra no relatório.


In [7]:
# Remove espaços nas bordas e transforma texto vazio em ausente.
text_columns = df.select_dtypes(include=["object", "string"]).columns
for column in text_columns:
    cleaned = df[column].astype("string").str.strip()
    df[column] = cleaned.mask(cleaned.eq(""), pd.NA)

# Contagens usadas no resumo final da limpeza.
invalid_date_count = 0
invalid_time_count = 0
numeric_conversion_report = {}
indicator_domain_issues = {}


### Jogo, tempo e situação de campo


In [8]:
# Date: data do jogo, sem horário. Armazenada como datetime64.
original = df["Date"]
datas = pd.to_datetime(original, errors="coerce")
invalid_date_count = int((original.notna() & datas.isna()).sum())
df["Date"] = datas


In [9]:
# GameID: identificador numérico do jogo (até 2017123115). Passa de int16 e cabe em int32.
original = df["GameID"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["GameID"] = int((original.notna() & serie.isna()).sum())
tipo = "Int32" if serie.isna().any() else "int32"
df["GameID"] = serie.astype(tipo)


In [10]:
# Drive: número da série, de 1 a 35. Cabe em int8.
original = df["Drive"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Drive"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["Drive"] = serie.astype(tipo)


In [11]:
# qtr: quarto da partida, de 1 a 5. Cabe em int8.
original = df["qtr"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["qtr"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["qtr"] = serie.astype(tipo)


In [12]:
# down: descida, de 1 a 4, com ausentes em eventos sem down. Cabe em int8.
original = df["down"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["down"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["down"] = serie.astype(tipo)


In [13]:
# time: relógio MM:SS do quarto vira segundos (0 a 900). Int16 anulável guarda o ausente.
original = df["time"]
texto = original.astype("string").str.strip()
interpretado = pd.to_timedelta("00:" + texto, errors="coerce")
invalid_time_count = int((original.notna() & interpretado.isna()).sum())
segundos = interpretado.dt.total_seconds()
numeric_conversion_report["time"] = invalid_time_count
df["time"] = segundos.astype("Int16")


In [14]:
# TimeUnder: minutos restantes no quarto, de 0 a 15. Cabe em int8.
original = df["TimeUnder"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["TimeUnder"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["TimeUnder"] = serie.astype(tipo)


In [15]:
# TimeSecs: segundos restantes no jogo, de 0 a 3600. Passa de int8 e cabe em int16.
original = df["TimeSecs"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["TimeSecs"] = int((original.notna() & serie.isna()).sum())
tipo = "Int16" if serie.isna().any() else "int16"
df["TimeSecs"] = serie.astype(tipo)


In [16]:
# PlayTimeDiff: segundos entre esta jogada e a anterior. Passa de int8 e cabe em int16.
original = df["PlayTimeDiff"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["PlayTimeDiff"] = int((original.notna() & serie.isna()).sum())
tipo = "Int16" if serie.isna().any() else "int16"
df["PlayTimeDiff"] = serie.astype(tipo)


In [17]:
# SideofField: sigla do lado do campo. Poucas equipes, então category vira dicionário no Parquet.
texto = df["SideofField"].astype("string").str.strip()
df["SideofField"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [18]:
# yrdln: linha de jardas, de 0 a 50. Cabe em int8.
original = df["yrdln"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["yrdln"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["yrdln"] = serie.astype(tipo)


In [19]:
# yrdline100: jardas até a end zone adversária, de 1 a 99. Cabe em int8.
original = df["yrdline100"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["yrdline100"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["yrdline100"] = serie.astype(tipo)


In [20]:
# ydstogo: jardas para a primeira descida. O domínio fica abaixo de 127, então cabe em int8.
original = df["ydstogo"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["ydstogo"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["ydstogo"] = serie.astype(tipo)


In [21]:
# ydsnet: jardas líquidas acumuladas na série. Uma série pode passar de 127 jardas, então o tipo é int16.
original = df["ydsnet"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["ydsnet"] = int((original.notna() & serie.isna()).sum())
tipo = "Int16" if serie.isna().any() else "int16"
df["ydsnet"] = serie.astype(tipo)


In [22]:
# GoalToGo: indicador 0/1 de situação and goal.
original = df["GoalToGo"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["GoalToGo"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["GoalToGo"] = valores.astype(tipo)


In [23]:
# posteam: sigla do time com a posse. Poucas equipes, então category.
texto = df["posteam"].astype("string").str.strip()
df["posteam"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [24]:
# DefensiveTeam: sigla do time na defesa. Poucas equipes, então category.
texto = df["DefensiveTeam"].astype("string").str.strip()
df["DefensiveTeam"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [25]:
# HomeTeam: sigla do mandante. Poucas equipes, então category.
texto = df["HomeTeam"].astype("string").str.strip()
df["HomeTeam"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [26]:
# AwayTeam: sigla do visitante. Poucas equipes, então category.
texto = df["AwayTeam"].astype("string").str.strip()
df["AwayTeam"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [27]:
# Season: ano da temporada (2009 a 2017). Passa de int8 e cabe em int16.
original = df["Season"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Season"] = int((original.notna() & serie.isna()).sum())
tipo = "Int16" if serie.isna().any() else "int16"
df["Season"] = serie.astype(tipo)


### Resultado geral da jogada


In [28]:
# desc: narração livre da jogada. Quase única por linha, então permanece string.
texto = df["desc"].astype("string").str.strip()
df["desc"] = texto.mask(texto.eq(""), pd.NA)


In [29]:
# PlayAttempted: indicador 0/1 de jogada contabilizada.
original = df["PlayAttempted"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["PlayAttempted"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["PlayAttempted"] = valores.astype(tipo)


In [30]:
# PlayType: tipo do evento, com poucas categorias (passe, corrida, punt, timeout etc.).
texto = df["PlayType"].astype("string").str.strip()
df["PlayType"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [31]:
# Yards.Gained: jardas da jogada. Uma jogada cabe em int8 (−128 a 127).
original = df["Yards.Gained"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Yards.Gained"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["Yards.Gained"] = serie.astype(tipo)


In [32]:
# FirstDown: indicador 0/1 de conversão de primeira descida.
original = df["FirstDown"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["FirstDown"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["FirstDown"] = valores.astype(tipo)


In [33]:
# sp: indicador 0/1 de jogada que gerou pontuação.
original = df["sp"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["sp"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["sp"] = valores.astype(tipo)


In [34]:
# Touchdown: indicador 0/1 de touchdown.
original = df["Touchdown"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Touchdown"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Touchdown"] = valores.astype(tipo)


In [35]:
# ExPointResult: resultado do ponto extra: Made, Missed, Blocked ou Aborted.
texto = df["ExPointResult"].astype("string").str.strip()
df["ExPointResult"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [36]:
# TwoPointConv: resultado da conversão de 2 pontos do ataque: Success ou Failure.
texto = df["TwoPointConv"].astype("string").str.strip()
df["TwoPointConv"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [37]:
# DefTwoPoint: resultado da conversão de 2 pontos da defesa: Success ou Failure.
texto = df["DefTwoPoint"].astype("string").str.strip()
df["DefTwoPoint"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [38]:
# Safety: indicador 0/1 de safety.
original = df["Safety"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Safety"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Safety"] = valores.astype(tipo)


In [39]:
# Onsidekick: indicador 0/1 de onside kick.
original = df["Onsidekick"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Onsidekick"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Onsidekick"] = valores.astype(tipo)


In [40]:
# PuntResult: resultado do punt: Clean ou Blocked.
texto = df["PuntResult"].astype("string").str.strip()
df["PuntResult"] = texto.mask(texto.eq(""), pd.NA).astype("category")


### Passe


In [41]:
# Passer: nome do passador. O mesmo jogador se repete, então category.
texto = df["Passer"].astype("string").str.strip()
df["Passer"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [42]:
# Passer_ID: identificador GSIS do passador, em texto. Category, sem converter o código para número.
texto = df["Passer_ID"].astype("string").str.strip()
df["Passer_ID"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [43]:
# PassAttempt: indicador 0/1 de tentativa de passe.
original = df["PassAttempt"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["PassAttempt"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["PassAttempt"] = valores.astype(tipo)


In [44]:
# PassOutcome: resultado do passe: Complete ou Incomplete.
texto = df["PassOutcome"].astype("string").str.strip()
df["PassOutcome"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [45]:
# PassLength: comprimento do passe: Short ou Deep.
texto = df["PassLength"].astype("string").str.strip()
df["PassLength"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [46]:
# AirYards: jardas que a bola percorreu no ar. O campo limita o valor a int8.
original = df["AirYards"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["AirYards"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["AirYards"] = serie.astype(tipo)


In [47]:
# YardsAfterCatch: jardas depois da recepção. Cabe em int8.
original = df["YardsAfterCatch"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["YardsAfterCatch"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["YardsAfterCatch"] = serie.astype(tipo)


In [48]:
# QBHit: indicador 0/1 de quarterback derrubado.
original = df["QBHit"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["QBHit"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["QBHit"] = valores.astype(tipo)


In [49]:
# PassLocation: lado do passe: left, middle ou right.
texto = df["PassLocation"].astype("string").str.strip()
df["PassLocation"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [50]:
# InterceptionThrown: indicador 0/1 de interceptação.
original = df["InterceptionThrown"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["InterceptionThrown"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["InterceptionThrown"] = valores.astype(tipo)


In [51]:
# Interceptor: nome de quem interceptou. Category pelo nome repetido.
texto = df["Interceptor"].astype("string").str.strip()
df["Interceptor"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [52]:
# Receiver: nome do recebedor alvo. Category pelo nome repetido.
texto = df["Receiver"].astype("string").str.strip()
df["Receiver"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [53]:
# Receiver_ID: identificador GSIS do recebedor, em texto. Category, sem converter para número.
texto = df["Receiver_ID"].astype("string").str.strip()
df["Receiver_ID"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [54]:
# Reception: indicador 0/1 de recepção.
original = df["Reception"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Reception"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Reception"] = valores.astype(tipo)


In [55]:
# Sack: indicador 0/1 de sack.
original = df["Sack"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Sack"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Sack"] = valores.astype(tipo)


### Corrida


In [56]:
# Rusher: nome do corredor. Category pelo nome repetido.
texto = df["Rusher"].astype("string").str.strip()
df["Rusher"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [57]:
# Rusher_ID: identificador GSIS do corredor, em texto. Category, sem converter para número.
texto = df["Rusher_ID"].astype("string").str.strip()
df["Rusher_ID"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [58]:
# RushAttempt: indicador 0/1 de corrida.
original = df["RushAttempt"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["RushAttempt"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["RushAttempt"] = valores.astype(tipo)


In [59]:
# RunLocation: lado da corrida: left, middle ou right.
texto = df["RunLocation"].astype("string").str.strip()
df["RunLocation"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [60]:
# RunGap: buraco da corrida: guard, tackle ou end.
texto = df["RunGap"].astype("string").str.strip()
df["RunGap"] = texto.mask(texto.eq(""), pd.NA).astype("category")


### Retorno, chute, fumble, revisão e penalidade


In [61]:
# ReturnResult: resultado do retorno: Fair Catch, Touchback ou Touchdown.
texto = df["ReturnResult"].astype("string").str.strip()
df["ReturnResult"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [62]:
# Returner: nome de quem retornou o chute. Category pelo nome repetido.
texto = df["Returner"].astype("string").str.strip()
df["Returner"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [63]:
# BlockingPlayer: nome de quem bloqueou o chute. Category pelo nome repetido.
texto = df["BlockingPlayer"].astype("string").str.strip()
df["BlockingPlayer"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [64]:
# Tackler1: nome do tackle principal. Category pelo nome repetido.
texto = df["Tackler1"].astype("string").str.strip()
df["Tackler1"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [65]:
# Tackler2: nome do segundo tackle. Category pelo nome repetido.
texto = df["Tackler2"].astype("string").str.strip()
df["Tackler2"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [66]:
# FieldGoalResult: resultado do field goal: Good, No Good ou Blocked.
texto = df["FieldGoalResult"].astype("string").str.strip()
df["FieldGoalResult"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [67]:
# FieldGoalDistance: distância do field goal em jardas inteiras (cerca de 18 a 70). Cabe em int8.
original = df["FieldGoalDistance"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["FieldGoalDistance"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["FieldGoalDistance"] = serie.astype(tipo)


In [68]:
# Fumble: indicador 0/1 de fumble.
original = df["Fumble"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Fumble"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Fumble"] = valores.astype(tipo)


In [69]:
# RecFumbTeam: sigla do time que recuperou o fumble.
texto = df["RecFumbTeam"].astype("string").str.strip()
df["RecFumbTeam"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [70]:
# RecFumbPlayer: nome de quem recuperou o fumble. Category pelo nome repetido.
texto = df["RecFumbPlayer"].astype("string").str.strip()
df["RecFumbPlayer"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [71]:
# Challenge.Replay: indicador 0/1 de revisão por replay.
original = df["Challenge.Replay"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Challenge.Replay"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Challenge.Replay"] = valores.astype(tipo)


In [72]:
# ChalReplayResult: resultado da revisão: Upheld ou Reversed.
texto = df["ChalReplayResult"].astype("string").str.strip()
df["ChalReplayResult"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [73]:
# Accepted.Penalty: indicador 0/1 de penalidade aceita.
original = df["Accepted.Penalty"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Accepted.Penalty"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Accepted.Penalty"] = valores.astype(tipo)


In [74]:
# PenalizedTeam: sigla do time penalizado.
texto = df["PenalizedTeam"].astype("string").str.strip()
df["PenalizedTeam"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [75]:
# PenaltyType: nome da falta. Conjunto fechado de tipos, então category.
texto = df["PenaltyType"].astype("string").str.strip()
df["PenaltyType"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [76]:
# PenalizedPlayer: nome do jogador penalizado. Category pelo nome repetido.
texto = df["PenalizedPlayer"].astype("string").str.strip()
df["PenalizedPlayer"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [77]:
# Penalty.Yards: jardas da penalidade. O valor fica abaixo de 127, então cabe em int8.
original = df["Penalty.Yards"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Penalty.Yards"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["Penalty.Yards"] = serie.astype(tipo)


### Placar e timeouts


In [78]:
# PosTeamScore: placar do time com a posse. Um placar de NFL cabe em int8.
original = df["PosTeamScore"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["PosTeamScore"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["PosTeamScore"] = serie.astype(tipo)


In [79]:
# DefTeamScore: placar da defesa. Um placar de NFL cabe em int8.
original = df["DefTeamScore"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["DefTeamScore"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["DefTeamScore"] = serie.astype(tipo)


In [80]:
# ScoreDiff: diferença de placar (ataque menos defesa). Cabe em int8.
original = df["ScoreDiff"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["ScoreDiff"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["ScoreDiff"] = serie.astype(tipo)


In [81]:
# AbsScoreDiff: módulo da diferença de placar. Cabe em int8.
original = df["AbsScoreDiff"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["AbsScoreDiff"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["AbsScoreDiff"] = serie.astype(tipo)


In [82]:
# Timeout_Indicator: indicador 0/1 de timeout cobrado.
original = df["Timeout_Indicator"]
valores = pd.to_numeric(original, errors="coerce")
fora = original.notna() & (valores.isna() | ~valores.isin([0, 1]))
quantidade_fora = int(fora.sum())
if quantidade_fora:
    indicator_domain_issues["Timeout_Indicator"] = quantidade_fora
else:
    tipo = "boolean" if valores.isna().any() else "bool"
    df["Timeout_Indicator"] = valores.astype(tipo)


In [83]:
# Timeout_Team: sigla do time que gastou o timeout.
texto = df["Timeout_Team"].astype("string").str.strip()
df["Timeout_Team"] = texto.mask(texto.eq(""), pd.NA).astype("category")


In [84]:
# posteam_timeouts_pre: timeouts restantes do time com a posse, de 0 a 3. Cabe em int8.
original = df["posteam_timeouts_pre"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["posteam_timeouts_pre"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["posteam_timeouts_pre"] = serie.astype(tipo)


In [85]:
# HomeTimeouts_Remaining_Pre: timeouts do mandante antes da jogada, de 0 a 3. Cabe em int8.
original = df["HomeTimeouts_Remaining_Pre"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["HomeTimeouts_Remaining_Pre"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["HomeTimeouts_Remaining_Pre"] = serie.astype(tipo)


In [86]:
# AwayTimeouts_Remaining_Pre: timeouts do visitante antes da jogada, de 0 a 3. Cabe em int8.
original = df["AwayTimeouts_Remaining_Pre"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["AwayTimeouts_Remaining_Pre"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["AwayTimeouts_Remaining_Pre"] = serie.astype(tipo)


In [87]:
# HomeTimeouts_Remaining_Post: timeouts do mandante depois da jogada, de 0 a 3. Cabe em int8.
original = df["HomeTimeouts_Remaining_Post"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["HomeTimeouts_Remaining_Post"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["HomeTimeouts_Remaining_Post"] = serie.astype(tipo)


In [88]:
# AwayTimeouts_Remaining_Post: timeouts do visitante depois da jogada, de 0 a 3. Cabe em int8.
original = df["AwayTimeouts_Remaining_Post"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["AwayTimeouts_Remaining_Post"] = int((original.notna() & serie.isna()).sum())
tipo = "Int8" if serie.isna().any() else "int8"
df["AwayTimeouts_Remaining_Post"] = serie.astype(tipo)


### Modelos: pontos esperados e probabilidade de vitória


In [89]:
# No_Score_Prob: probabilidade de ninguém pontuar no restante do tempo.
original = df["No_Score_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["No_Score_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["No_Score_Prob"] = serie.astype(tipo)


In [90]:
# Opp_Field_Goal_Prob: probabilidade de a defesa marcar o próximo field goal.
original = df["Opp_Field_Goal_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Opp_Field_Goal_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Opp_Field_Goal_Prob"] = serie.astype(tipo)


In [91]:
# Opp_Safety_Prob: probabilidade de a defesa marcar o próximo safety.
original = df["Opp_Safety_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Opp_Safety_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Opp_Safety_Prob"] = serie.astype(tipo)


In [92]:
# Opp_Touchdown_Prob: probabilidade de a defesa marcar o próximo touchdown.
original = df["Opp_Touchdown_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Opp_Touchdown_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Opp_Touchdown_Prob"] = serie.astype(tipo)


In [93]:
# Field_Goal_Prob: probabilidade de o ataque marcar o próximo field goal.
original = df["Field_Goal_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Field_Goal_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Field_Goal_Prob"] = serie.astype(tipo)


In [94]:
# Safety_Prob: probabilidade de o ataque marcar o próximo safety.
original = df["Safety_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Safety_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Safety_Prob"] = serie.astype(tipo)


In [95]:
# Touchdown_Prob: probabilidade de o ataque marcar o próximo touchdown.
original = df["Touchdown_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Touchdown_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Touchdown_Prob"] = serie.astype(tipo)


In [96]:
# ExPoint_Prob: probabilidade de acertar o ponto extra.
original = df["ExPoint_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["ExPoint_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["ExPoint_Prob"] = serie.astype(tipo)


In [97]:
# TwoPoint_Prob: probabilidade de converter os 2 pontos.
original = df["TwoPoint_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["TwoPoint_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["TwoPoint_Prob"] = serie.astype(tipo)


In [98]:
# ExpPts: pontos esperados do ataque antes da jogada.
original = df["ExpPts"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["ExpPts"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["ExpPts"] = serie.astype(tipo)


In [99]:
# EPA: variação dos pontos esperados causada pela jogada.
original = df["EPA"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["EPA"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["EPA"] = serie.astype(tipo)


In [100]:
# airEPA: parte do EPA atribuída às jardas no ar.
original = df["airEPA"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["airEPA"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["airEPA"] = serie.astype(tipo)


In [101]:
# yacEPA: parte do EPA atribuída às jardas depois da recepção.
original = df["yacEPA"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["yacEPA"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["yacEPA"] = serie.astype(tipo)


In [102]:
# Home_WP_pre: probabilidade de vitória do mandante antes da jogada.
original = df["Home_WP_pre"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Home_WP_pre"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Home_WP_pre"] = serie.astype(tipo)


In [103]:
# Away_WP_pre: probabilidade de vitória do visitante antes da jogada.
original = df["Away_WP_pre"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Away_WP_pre"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Away_WP_pre"] = serie.astype(tipo)


In [104]:
# Home_WP_post: probabilidade de vitória do mandante depois da jogada.
original = df["Home_WP_post"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Home_WP_post"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Home_WP_post"] = serie.astype(tipo)


In [105]:
# Away_WP_post: probabilidade de vitória do visitante depois da jogada.
original = df["Away_WP_post"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Away_WP_post"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Away_WP_post"] = serie.astype(tipo)


In [106]:
# Win_Prob: probabilidade de vitória do time com a posse antes da jogada.
original = df["Win_Prob"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["Win_Prob"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["Win_Prob"] = serie.astype(tipo)


In [107]:
# WPA: variação da probabilidade de vitória causada pela jogada.
original = df["WPA"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["WPA"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["WPA"] = serie.astype(tipo)


In [108]:
# airWPA: parte do WPA atribuída às jardas no ar.
original = df["airWPA"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["airWPA"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["airWPA"] = serie.astype(tipo)


In [109]:
# yacWPA: parte do WPA atribuída às jardas depois da recepção.
original = df["yacWPA"]
serie = pd.to_numeric(original, errors="coerce")
numeric_conversion_report["yacWPA"] = int((original.notna() & serie.isna()).sum())
tipo = "Float32" if serie.isna().any() else "float32"
df["yacWPA"] = serie.astype(tipo)


### Relatório da tipificação


In [110]:
display(pd.Series(numeric_conversion_report, name="valores numéricos inválidos"))
display(pd.Series(indicator_domain_issues, name="indicadores fora de {0, 1}"))
print(f"Valores não ausentes de Date que não puderam ser interpretados: {invalid_date_count:,}")
print(f"Valores não ausentes de time que não puderam ser interpretados: {invalid_time_count:,}")


GameID                         0
Drive                          0
qtr                            0
down                           0
time                           0
TimeUnder                      0
TimeSecs                       0
PlayTimeDiff                   0
yrdln                          0
yrdline100                     0
ydstogo                        0
ydsnet                         0
Season                         0
Yards.Gained                   0
AirYards                       0
YardsAfterCatch                0
FieldGoalDistance              0
Penalty.Yards                  0
PosTeamScore                   0
DefTeamScore                   0
ScoreDiff                      0
AbsScoreDiff                   0
posteam_timeouts_pre           0
HomeTimeouts_Remaining_Pre     0
AwayTimeouts_Remaining_Pre     0
HomeTimeouts_Remaining_Post    0
AwayTimeouts_Remaining_Post    0
No_Score_Prob                  0
Opp_Field_Goal_Prob            0
Opp_Safety_Prob                0
Opp_Touchd

Series([], Name: indicadores fora de {0, 1}, dtype: object)

Valores não ausentes de Date que não puderam ser interpretados: 0
Valores não ausentes de time que não puderam ser interpretados: 0


## 4. Valores ausentes

Não se imputam valores nos atributos das jogadas: sem contexto do significado de cada ausência, substituir por zero, média ou moda pode criar eventos fictícios (por exemplo, confundir “não se aplica” com uma medição igual a zero). Também não se excluem registros apenas por conterem ausências. Textos vazios/brancos são normalizados como ausentes. Datas e tempos inválidos são convertidos para ausentes e quantificados. Para a coluna numérica `time`, os ausentes permanecem como nulos no tipo anulável `Int16`; não é possível inserir `""` mantendo essa coluna como inteiro. No Parquet, nulos são representados como valores null com metadados de validade, preservando o tipo numérico e evitando converter a coluna em texto, o que poderia aumentar o arquivo. A tabela abaixo permite comparar as ausências antes e depois do tratamento.

In [111]:
# Compara contagem e percentual de ausências antes e depois da padronização.
missing_after_typing = pd.DataFrame({
    "ausentes antes": df_original.isna().sum(),
    "ausentes depois": df.isna().sum(),
    "percentual depois": df.isna().mean().mul(100),
}).sort_values("ausentes depois", ascending=False)
display(missing_after_typing)
print(f"Valores inválidos de data convertidos para ausentes: {invalid_date_count:,}")
print(f"Valores inválidos de time convertidos para ausentes: {invalid_time_count:,}")

,ausentes antes,ausentes depois,percentual depois
DefTwoPoint,407664,407664,99.994113
BlockingPlayer,407571,407571,99.971302
TwoPointConv,407083,407083,99.851602
ChalReplayResult,404286,404286,99.165538
RecFumbPlayer,403315,403315,98.927366
...,...,...,...
HomeTeam,0,0,0.000000
Timeout_Indicator,0,0,0.000000
TwoPoint_Prob,0,0,0.000000
ExPoint_Prob,0,0,0.000000


Valores inválidos de data convertidos para ausentes: 0
Valores inválidos de time convertidos para ausentes: 0


## 5. Duplicidades e verificações de consistência

Duplicatas serão definidas como linhas idênticas em todas as colunas e removidas mantendo a primeira ocorrência. Verificações de domínio simples servem para sinalizar itens que merecem revisão, não para presumir que os dados sejam incorretos. O trimestre pode incluir overtime (até 5 no domínio esperado desta fonte), o down deve estar entre 1 e 4 e `ydstogo` não deve ser negativo. Nenhum registro será excluído por esses alertas.

In [112]:
# Registra duplicatas integrais e remove apenas repetições exatas.
duplicate_count_removed = int(df.duplicated().sum())
df = df.drop_duplicates(keep="first").reset_index(drop=True)

# Quantifica valores fora dos domínios simples; ausências não são contadas.
domain_issue_counts = {}
domain_rules = {
    "qtr fora de 1..5": ("qtr", lambda values: ~values.between(1, 5)),
    "down fora de 1..4": ("down", lambda values: ~values.between(1, 4)),
    "ydstogo negativo": ("ydstogo", lambda values: values.lt(0)),
}
for label, (column, invalid_rule) in domain_rules.items():
    if column in df.columns:
        numeric_values = pd.to_numeric(df[column], errors="coerce")
        domain_issue_counts[label] = int((numeric_values.notna() & invalid_rule(numeric_values)).sum())
display(pd.Series(domain_issue_counts, name="quantidade sinalizada"))
print(f"Duplicatas integrais removidas: {duplicate_count_removed:,}")
print(f"Dimensões após limpeza: {df.shape[0]:,} linhas × {df.shape[1]:,} colunas")

qtr fora de 1..5     0
down fora de 1..4    0
ydstogo negativo     0
Name: quantidade sinalizada, dtype: int64

Duplicatas integrais removidas: 0
Dimensões após limpeza: 407,688 linhas × 102 colunas


## 6. Resumo da limpeza e dos tipos finais

A tabela resume alterações estruturais e a completude final. As inconsistências sinalizadas permanecem no DataFrame para evitar correções sem evidência ou regra de negócio.

In [113]:
# Consolida indicadores finais antes da persistência.
quality_summary = pd.Series({
    "linhas originais": len(df_original),
    "linhas finais": len(df),
    "colunas": df.shape[1],
    "duplicatas exatas removidas": duplicate_count_removed,
    "valores ausentes finais": int(df.isna().sum().sum()),
    "valores de Date inválidos": invalid_date_count,
    "indicadores com valores fora de {0, 1}": len(indicator_domain_issues),
    "regras de domínio com ocorrências": sum(count > 0 for count in domain_issue_counts.values()),
}, name="resultado")
display(quality_summary.to_frame())
display(pd.DataFrame({"tipo final": df.dtypes.astype(str), "ausentes": df.isna().sum()}))

,resultado
linhas originais,407688
linhas finais,407688
colunas,102
duplicatas exatas removidas,0
valores ausentes finais,11505187
valores de Date inválidos,0
"indicadores com valores fora de {0, 1}",0
regras de domínio com ocorrências,0


,tipo final,ausentes
Date,datetime64[us],0
GameID,int32,0
Drive,int8,0
qtr,int8,0
down,Int8,61154
...,...,...
Win_Prob,Float32,25009
WPA,Float32,5541
airWPA,Float32,248501
yacWPA,Float32,248762


## 7. Armazenamento em Parquet e comparação de tamanho

O arquivo tratado será salvo no diretório de trabalho atual. Parquet é colunar e costuma ser menor que CSV, mas o resultado depende dos dados, compressão e tipos. A comparação será feita em bytes e também apresentada em MB e percentual; não se presume previamente qual arquivo será menor.

In [114]:
# Grava sem o índice do pandas, pois ele não representa um atributo do dataset.
parquet_path = Path.cwd() / "NFL_play_by_play_tratado.parquet"
df.to_parquet(parquet_path, index=False)

# Compara o tamanho físico do CSV de entrada com o Parquet gerado.
csv_size_bytes = csv_path.stat().st_size
parquet_size_bytes = parquet_path.stat().st_size
size_reduction_percent = (1 - parquet_size_bytes / csv_size_bytes) * 100 if csv_size_bytes else float("nan")
size_comparison = pd.DataFrame({
    "arquivo": [str(csv_path), str(parquet_path)],
    "tamanho (bytes)": [csv_size_bytes, parquet_size_bytes],
    "tamanho (MB)": [csv_size_bytes / (1024 ** 2), parquet_size_bytes / (1024 ** 2)],
})
display(size_comparison)
print(f"Redução do tamanho: {size_reduction_percent:.2f}% (valor negativo indica aumento).")

,arquivo,tamanho (bytes),tamanho (MB)
0,C:\Users\41228408807\.cache\kagglehub\datasets...,275530460,262.766323
1,c:\Users\41228408807\Documents\GitHub\uniSENAI...,56917805,54.281049


Redução do tamanho: 79.34% (valor negativo indica aumento).


## 8. Validação após armazenamento

O Parquet é reaberto e comparado com o DataFrame tratado, incluindo ordem das colunas e conteúdo, com ausências equivalentes. O índice não é comparado porque foi deliberadamente excluído do arquivo. A mensagem de validação só aparece se as verificações forem aprovadas; qualquer divergência gera uma exceção explícita.

In [115]:
# Recarrega o arquivo persistido e valida dimensões, colunas, tipos e valores.
df_reloaded = pd.read_parquet(parquet_path)
assert df_reloaded.shape == df.shape, "As dimensões mudaram após a gravação Parquet."
assert df_reloaded.columns.tolist() == df.columns.tolist(), "A ordem ou os nomes das colunas mudaram."
assert_frame_equal(
    df.reset_index(drop=True),
    df_reloaded.reset_index(drop=True),
    check_dtype=True,
    check_categorical=False,
    check_exact=False,
    check_names=True,
)
print("Validação aprovada: dimensões, colunas e dados foram preservados no Parquet.")

Validação aprovada: dimensões, colunas e dados foram preservados no Parquet.


## 9. Análise final

O resumo final é produzido a partir dos resultados calculados durante a execução, sem números pré-preenchidos. Ele reúne as principais mudanças, caminhos e comparação de armazenamento.

In [116]:
# Apresenta os principais resultados para documentar a execução.
final_summary = {
    "CSV de origem": str(csv_path),
    "Parquet tratado": str(parquet_path.resolve()),
    "linhas antes": len(df_original),
    "linhas após limpeza": len(df),
    "colunas preservadas": df.shape[1],
    "duplicatas integrais removidas": duplicate_count_removed,
    "ausências mantidas como nulos no resultado": int(df.isna().sum().sum()),
    "datas inválidas convertidas para NaT": invalid_date_count,
    "tempos de jogo inválidos convertidos para ausentes": invalid_time_count,
    "time armazenado como segundos em Int16 anulável": "sim" if "time" in df.columns else "coluna ausente",
    "conversões numéricas malsucedidas": sum(numeric_conversion_report.values()),
    "valores fora do domínio de indicadores": sum(indicator_domain_issues.values()),
    "ocorrências sinalizadas pelas regras de domínio": sum(domain_issue_counts.values()),
    "tamanho CSV (MB)": round(csv_size_bytes / (1024 ** 2), 2),
    "tamanho Parquet (MB)": round(parquet_size_bytes / (1024 ** 2), 2),
    "redução de tamanho (%)": round(size_reduction_percent, 2),
    "validação Parquet": "aprovada se esta célula foi concluída sem exceção",
}
display(pd.Series(final_summary, name="resultado final").to_frame())
print(
    "Tratamento concluído: tipos semânticos ajustados, vazios padronizados, "
    "duplicatas exatas removidas, anomalias reportadas sem exclusão e Parquet validado."
)

,resultado final
CSV de origem,C:\Users\41228408807\.cache\kagglehub\datasets...
Parquet tratado,C:\Users\41228408807\Documents\GitHub\uniSENAI...
linhas antes,407688
linhas após limpeza,407688
colunas preservadas,102
duplicatas integrais removidas,0
ausências mantidas como nulos no resultado,11505187
datas inválidas convertidas para NaT,0
tempos de jogo inválidos convertidos para ausentes,0
time armazenado como segundos em Int16 anulável,sim


Tratamento concluído: tipos semânticos ajustados, vazios padronizados, duplicatas exatas removidas, anomalias reportadas sem exclusão e Parquet validado.
